In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 300)
pd.set_option('display.width', 220)

DATA_DIR = Path('data')
OUT_M = DATA_DIR / 'm_data.csv'
OUT_W = DATA_DIR / 'w_data.csv'
OUT_FULL = Path('fulldata.csv')
OUT_FULL_COMPAT = Path('fulldataset.csv')

if not DATA_DIR.exists():
    raise FileNotFoundError(f'Nie znaleziono katalogu z danymi: {DATA_DIR.resolve()}')


def read_csv_safe(filename, required=False):
    path = DATA_DIR / filename
    if path.exists():
        print(f'[read] {filename}')
        return pd.read_csv(path)
    if required:
        raise FileNotFoundError(f'Brakuje wymaganego pliku: {path}')
    print(f'[skip] brak opcjonalnego pliku: {filename}')
    return None

In [3]:
# required core files
m_regular = read_csv_safe('MRegularSeasonDetailedResults.csv', required=True)
w_regular = read_csv_safe('WRegularSeasonDetailedResults.csv', required=True)
m_tourney = read_csv_safe('MNCAATourneyDetailedResults.csv', required=True)
w_tourney = read_csv_safe('WNCAATourneyDetailedResults.csv', required=True)

# optional enrichment files
m_seeds = read_csv_safe('MNCAATourneySeeds.csv')
w_seeds = read_csv_safe('WNCAATourneySeeds.csv')
m_teams = read_csv_safe('MTeams.csv')
w_teams = read_csv_safe('WTeams.csv')
m_seasons = read_csv_safe('MSeasons.csv')
w_seasons = read_csv_safe('WSeasons.csv')
conferences = read_csv_safe('Conferences.csv')
m_team_conf = read_csv_safe('MTeamConferences.csv')
w_team_conf = read_csv_safe('WTeamConferences.csv')
m_game_cities = read_csv_safe('MGameCities.csv')
w_game_cities = read_csv_safe('WGameCities.csv')
cities = read_csv_safe('Cities.csv')
m_conf_tourney = read_csv_safe('MConferenceTourneyGames.csv')
w_conf_tourney = read_csv_safe('WConferenceTourneyGames.csv')
m_coaches = read_csv_safe('MTeamCoaches.csv')
m_massey = read_csv_safe('MMasseyOrdinals.csv')

[read] MRegularSeasonDetailedResults.csv
[read] WRegularSeasonDetailedResults.csv
[read] MNCAATourneyDetailedResults.csv
[read] WNCAATourneyDetailedResults.csv
[read] MNCAATourneySeeds.csv
[read] WNCAATourneySeeds.csv
[read] MTeams.csv
[read] WTeams.csv
[read] MSeasons.csv
[read] WSeasons.csv
[read] Conferences.csv
[read] MTeamConferences.csv
[read] WTeamConferences.csv
[read] MGameCities.csv
[read] WGameCities.csv
[read] Cities.csv
[read] MConferenceTourneyGames.csv
[read] WConferenceTourneyGames.csv
[read] MTeamCoaches.csv
[skip] brak opcjonalnego pliku: MMasseyOrdinals.csv


In [4]:
def add_game_id(df, gender):
    out = df.copy()
    low_id = np.minimum(out['WTeamID'].astype(int), out['LTeamID'].astype(int))
    high_id = np.maximum(out['WTeamID'].astype(int), out['LTeamID'].astype(int))
    out['GameID'] = (
        gender + '_'
        + out['Season'].astype(str) + '_'
        + out['DayNum'].astype(str) + '_'
        + low_id.astype(str) + '_'
        + high_id.astype(str)
    )
    return out


def add_basic_flags(df):
    out = df.copy()
    out['NeutralSite'] = (out['WLoc'] == 'N').astype('int8')
    out['WHome'] = (out['WLoc'] == 'H').astype('int8')
    out['WHomeAway'] = out['WLoc']
    out['LHome'] = (out['WLoc'] == 'A').astype('int8')
    out['WAway'] = (out['WLoc'] == 'A').astype('int8')
    out['LAway'] = (out['WLoc'] == 'H').astype('int8')
    out['ScoreDiff'] = out['WScore'] - out['LScore']
    out['TotalScore'] = out['WScore'] + out['LScore']
    return out


def add_season_info(df, seasons):
    if seasons is None:
        return df
    keep = [c for c in ['Season', 'DayZero', 'RegionW', 'RegionX', 'RegionY', 'RegionZ'] if c in seasons.columns]
    if not keep:
        return df
    meta = seasons[keep].copy()
    if 'DayZero' in meta.columns:
        meta['DayZero'] = pd.to_datetime(meta['DayZero'], errors='coerce')
    out = df.merge(meta, on='Season', how='left')
    if 'DayZero' in out.columns:
        out['GameDate'] = out['DayZero'] + pd.to_timedelta(out['DayNum'], unit='D')
    return out


def add_team_info(df, teams, side_prefix):
    if teams is None:
        return df
    rename_map = {}
    for old, new in {
        'TeamID': f'{side_prefix}TeamID',
        'TeamName': f'{side_prefix}TeamName',
        'FirstD1Season': f'{side_prefix}FirstD1Season',
        'LastD1Season': f'{side_prefix}LastD1Season',
    }.items():
        if old in teams.columns:
            rename_map[old] = new
    if not rename_map:
        return df
    meta = teams[list(rename_map.keys())].rename(columns=rename_map).copy()
    out = df.merge(meta, on=f'{side_prefix}TeamID', how='left')
    if f'{side_prefix}FirstD1Season' in out.columns:
        out[f'{side_prefix}YearsInD1'] = out['Season'] - out[f'{side_prefix}FirstD1Season']
    return out


def add_seed_info(df, seeds, side_prefix):
    if seeds is None:
        return df
    meta = seeds.rename(columns={'TeamID': f'{side_prefix}TeamID', 'Seed': f'{side_prefix}Seed'}).copy()
    keep = [c for c in ['Season', f'{side_prefix}TeamID', f'{side_prefix}Seed'] if c in meta.columns]
    meta = meta[keep]
    out = df.merge(meta, on=['Season', f'{side_prefix}TeamID'], how='left')
    seed_col = f'{side_prefix}Seed'
    if seed_col in out.columns:
        seed_str = out[seed_col].astype('string')
        out[f'{side_prefix}SeedRegion'] = seed_str.str.extract(r'^([A-Z])', expand=False)
        out[f'{side_prefix}SeedNum'] = pd.to_numeric(
            seed_str.str.extract(r'^[A-Z](\d{2})', expand=False), errors='coerce'
        ).astype('Int64')
        out[f'{side_prefix}SeedPlayIn'] = seed_str.str.extract(r'^[A-Z]\d{2}([A-Za-z]+)$', expand=False)
    return out


def add_conference_info(df, team_conf, conferences, side_prefix):
    if team_conf is None:
        return df
    conf = team_conf.copy()
    rename_map = {'TeamID': f'{side_prefix}TeamID', 'ConfAbbrev': f'{side_prefix}ConfAbbrev'}
    conf = conf.rename(columns=rename_map)
    keep = [c for c in ['Season', f'{side_prefix}TeamID', f'{side_prefix}ConfAbbrev'] if c in conf.columns]
    conf = conf[keep].copy()
    if conferences is not None and 'ConfAbbrev' in conferences.columns:
        conf_names = conferences.copy()
        conf_names = conf_names.rename(columns={'ConfAbbrev': f'{side_prefix}ConfAbbrev', 'Description': f'{side_prefix}ConfName'})
        keep_conf = [c for c in [f'{side_prefix}ConfAbbrev', f'{side_prefix}ConfName'] if c in conf_names.columns]
        conf_names = conf_names[keep_conf].drop_duplicates()
        conf = conf.merge(conf_names, on=f'{side_prefix}ConfAbbrev', how='left')
    return df.merge(conf, on=['Season', f'{side_prefix}TeamID'], how='left')


def add_game_city_info(df, game_cities, cities):
    if game_cities is None:
        return df
    gc = game_cities.copy()
    keep = [c for c in ['Season', 'DayNum', 'WTeamID', 'LTeamID', 'CRType', 'CityID'] if c in gc.columns]
    gc = gc[keep].drop_duplicates().copy()
    out = df.merge(gc, on=['Season', 'DayNum', 'WTeamID', 'LTeamID'], how='left')
    if cities is not None and 'CityID' in out.columns:
        city_meta = cities.copy()
        rename_map = {'City': 'GameCity', 'State': 'GameState'}
        city_meta = city_meta.rename(columns=rename_map)
        keep_city = [c for c in ['CityID', 'GameCity', 'GameState'] if c in city_meta.columns]
        city_meta = city_meta[keep_city].drop_duplicates()
        out = out.merge(city_meta, on='CityID', how='left')
    return out


def add_conference_tourney_flag(df, conf_tourney):
    out = df.copy()
    out['isConferenceTourney'] = 0
    if conf_tourney is None:
        return out
    key_cols = [c for c in ['Season', 'DayNum', 'WTeamID', 'LTeamID'] if c in conf_tourney.columns]
    if len(key_cols) < 4:
        return out
    marker = conf_tourney[key_cols].drop_duplicates().copy()
    marker['isConferenceTourney'] = 1
    out = out.drop(columns=['isConferenceTourney'], errors='ignore').merge(marker, on=key_cols, how='left')
    out['isConferenceTourney'] = out['isConferenceTourney'].fillna(0).astype('int8')
    return out


def add_same_conference_flag(df):
    out = df.copy()
    if 'WConfAbbrev' in out.columns and 'LConfAbbrev' in out.columns:
        out['SameConference'] = (
            out['WConfAbbrev'].notna()
            & out['LConfAbbrev'].notna()
            & (out['WConfAbbrev'] == out['LConfAbbrev'])
        ).astype('int8')
    return out

In [9]:
def build_massey_snapshot(massey):
    if massey is None:
        return None
    req = {'Season', 'RankingDayNum', 'SystemName', 'TeamID', 'OrdinalRank'}
    if not req.issubset(set(massey.columns)):
        return None

    mm = massey.copy()
    mm = mm[list(req)].dropna(subset=['Season', 'RankingDayNum', 'TeamID', 'OrdinalRank']).copy()
    mm['Season'] = mm['Season'].astype(int)
    mm['RankingDayNum'] = mm['RankingDayNum'].astype(int)
    mm['TeamID'] = mm['TeamID'].astype(int)
    mm['OrdinalRank'] = pd.to_numeric(mm['OrdinalRank'], errors='coerce')
    mm = mm.dropna(subset=['OrdinalRank'])

    agg = (
        mm.groupby(['Season', 'RankingDayNum', 'TeamID'], as_index=False)
          .agg(
              MasseyMeanRank=('OrdinalRank', 'mean'),
              MasseyMedianRank=('OrdinalRank', 'median'),
              MasseyBestRank=('OrdinalRank', 'min'),
              MasseyWorstRank=('OrdinalRank', 'max'),
              MasseySystemCount=('OrdinalRank', 'size'),
              MasseyTop10Count=('OrdinalRank', lambda s: int((s <= 10).sum())),
              MasseyTop25Count=('OrdinalRank', lambda s: int((s <= 25).sum())),
          )
    )

    preferred_systems = ['POM', 'SAG', 'MOR', 'RPI', 'WLK', 'DOK', 'COL', 'MAS', 'BOB']
    available = [s for s in preferred_systems if s in set(mm['SystemName'].unique())]
    if available:
        pivot = (
            mm[mm['SystemName'].isin(available)]
            .pivot_table(index=['Season', 'RankingDayNum', 'TeamID'], columns='SystemName', values='OrdinalRank', aggfunc='first')
            .reset_index()
        )
        pivot.columns = [
            c if c in ['Season', 'RankingDayNum', 'TeamID'] else f'Massey_{c}'
            for c in pivot.columns
        ]
        agg = agg.merge(pivot, on=['Season', 'RankingDayNum', 'TeamID'], how='left')

    return agg


def add_latest_massey(df, massey_snapshot, side_prefix):
    if massey_snapshot is None:
        return df

    left = df.copy()
    right = massey_snapshot.copy()

    left[f'{side_prefix}RankKey'] = (
        left['Season'].astype(str) + '_' + left[f'{side_prefix}TeamID'].astype(str)
    )
    right['RankKey'] = (
        right['Season'].astype(str) + '_' + right['TeamID'].astype(str)
    )

    feature_cols = [c for c in right.columns if c not in ['Season', 'TeamID', 'RankKey']]
    rename_map = {'RankingDayNum': f'{side_prefix}MasseyDayNum'}
    rename_map.update({
        c: f'{side_prefix}{c}'
        for c in feature_cols
        if c != 'RankingDayNum'
    })
    right = right.rename(columns=rename_map)

    right_sort_col = f'{side_prefix}MasseyDayNum'
    keep_cols = ['RankKey', right_sort_col] + [
        c for c in right.columns
        if c.startswith(side_prefix) and c != f'{side_prefix}TeamID'
    ]
    right = right[keep_cols].copy()

    # KLUCZOWE: sortujemy najpierw po kolumnie "on", potem po "by"
    left = left.sort_values(['DayNum', f'{side_prefix}RankKey', 'GameID']).reset_index(drop=True)
    right = right.sort_values([right_sort_col, 'RankKey']).reset_index(drop=True)

    out = pd.merge_asof(
        left,
        right,
        left_on='DayNum',
        right_on=right_sort_col,
        left_by=f'{side_prefix}RankKey',
        right_by='RankKey',
        direction='backward'
    )

    out = out.drop(columns=[f'{side_prefix}RankKey', 'RankKey'], errors='ignore')
    return out.sort_values(['Season', 'DayNum', 'GameID']).reset_index(drop=True)


def add_active_coach(df, coaches, side_prefix):
    if coaches is None:
        return df

    required = {'Season', 'TeamID', 'FirstDayNum', 'LastDayNum', 'CoachName'}
    if not required.issubset(set(coaches.columns)):
        return df

    left = df.copy()
    right = coaches.copy()

    left[f'{side_prefix}CoachKey'] = (
        left['Season'].astype(str) + '_' + left[f'{side_prefix}TeamID'].astype(str)
    )
    right[f'{side_prefix}CoachKey'] = (
        right['Season'].astype(str) + '_' + right['TeamID'].astype(str)
    )

    right = right.rename(columns={
        'FirstDayNum': f'{side_prefix}CoachStartDayNum',
        'LastDayNum': f'{side_prefix}CoachEndDayNum',
        'CoachName': f'{side_prefix}CoachName'
    })

    right = right[[
        f'{side_prefix}CoachKey',
        f'{side_prefix}CoachStartDayNum',
        f'{side_prefix}CoachEndDayNum',
        f'{side_prefix}CoachName'
    ]].copy()

    # KLUCZOWE: sortujemy najpierw po kolumnie "on", potem po "by"
    left = left.sort_values(['DayNum', f'{side_prefix}CoachKey', 'GameID']).reset_index(drop=True)
    right = right.sort_values([f'{side_prefix}CoachStartDayNum', f'{side_prefix}CoachKey']).reset_index(drop=True)

    out = pd.merge_asof(
        left,
        right,
        left_on='DayNum',
        right_on=f'{side_prefix}CoachStartDayNum',
        left_by=f'{side_prefix}CoachKey',
        right_by=f'{side_prefix}CoachKey',
        direction='backward'
    )

    end_col = f'{side_prefix}CoachEndDayNum'
    name_col = f'{side_prefix}CoachName'
    start_col = f'{side_prefix}CoachStartDayNum'

    if end_col in out.columns:
        invalid = out[end_col].notna() & (out['DayNum'] > out[end_col])
        out.loc[invalid, [start_col, end_col, name_col]] = pd.NA

    out = out.drop(columns=[f'{side_prefix}CoachKey'], errors='ignore')
    return out.sort_values(['Season', 'DayNum', 'GameID']).reset_index(drop=True)

In [10]:
m_massey_snapshot = build_massey_snapshot(m_massey)
if m_massey_snapshot is not None:
    print('Massey snapshot shape:', m_massey_snapshot.shape)
    display(m_massey_snapshot.head())
else:
    print('Brak snapshotu Massey - kolumny rankingowe będą pominięte.')

Brak snapshotu Massey - kolumny rankingowe będą pominięte.


In [11]:
def build_gender_dataset(
    gender,
    regular,
    tourney,
    seeds=None,
    teams=None,
    seasons=None,
    team_conf=None,
    conferences=None,
    game_cities=None,
    cities=None,
    conf_tourney=None,
    coaches=None,
    massey_snapshot=None,
):
    regular = regular.copy()
    tourney = tourney.copy()

    regular['Gender'] = gender
    tourney['Gender'] = gender
    regular['GameType'] = 'RegularSeason'
    tourney['GameType'] = 'NCAATourney'
    regular['isTourney'] = 0
    tourney['isTourney'] = 1
    regular['isRegularSeason'] = 1
    tourney['isRegularSeason'] = 0

    full = pd.concat([regular, tourney], ignore_index=True)
    full = add_game_id(full, gender)
    full = add_basic_flags(full)
    full = add_season_info(full, seasons)
    full = add_team_info(full, teams, 'W')
    full = add_team_info(full, teams, 'L')
    full = add_seed_info(full, seeds, 'W')
    full = add_seed_info(full, seeds, 'L')
    full = add_conference_info(full, team_conf, conferences, 'W')
    full = add_conference_info(full, team_conf, conferences, 'L')
    full = add_same_conference_flag(full)
    full = add_game_city_info(full, game_cities, cities)
    full = add_conference_tourney_flag(full, conf_tourney)

    if gender == 'M':
        full = add_active_coach(full, coaches, 'W')
        full = add_active_coach(full, coaches, 'L')
        full = add_latest_massey(full, massey_snapshot, 'W')
        full = add_latest_massey(full, massey_snapshot, 'L')

    full = full.sort_values(['Season', 'DayNum', 'GameID']).reset_index(drop=True)

    preferred_front = [
        'GameID', 'Gender', 'Season', 'DayNum', 'GameDate', 'GameType', 'isRegularSeason', 'isTourney', 'isConferenceTourney',
        'WTeamID', 'WTeamName', 'LTeamID', 'LTeamName',
        'WConfAbbrev', 'WConfName', 'LConfAbbrev', 'LConfName', 'SameConference',
        'WSeed', 'WSeedNum', 'WSeedRegion', 'LSeed', 'LSeedNum', 'LSeedRegion',
        'WLoc', 'NeutralSite', 'WHome', 'LHome', 'WAway', 'LAway',
        'CityID', 'GameCity', 'GameState', 'CRType',
        'WCoachName', 'LCoachName',
        'WMasseyMeanRank', 'WMasseyMedianRank', 'WMasseyBestRank', 'WMasseyWorstRank',
        'LMasseyMeanRank', 'LMasseyMedianRank', 'LMasseyBestRank', 'LMasseyWorstRank',
        'WScore', 'LScore', 'ScoreDiff', 'TotalScore', 'NumOT'
    ]
    preferred_front = [c for c in preferred_front if c in full.columns]
    remaining = [c for c in full.columns if c not in preferred_front]
    full = full[preferred_front + remaining]
    return full

In [12]:
m_data = build_gender_dataset(
    gender='M',
    regular=m_regular,
    tourney=m_tourney,
    seeds=m_seeds,
    teams=m_teams,
    seasons=m_seasons,
    team_conf=m_team_conf,
    conferences=conferences,
    game_cities=m_game_cities,
    cities=cities,
    conf_tourney=m_conf_tourney,
    coaches=m_coaches,
    massey_snapshot=m_massey_snapshot,
)

w_data = build_gender_dataset(
    gender='W',
    regular=w_regular,
    tourney=w_tourney,
    seeds=w_seeds,
    teams=w_teams,
    seasons=w_seasons,
    team_conf=w_team_conf,
    conferences=conferences,
    game_cities=w_game_cities,
    cities=cities,
    conf_tourney=w_conf_tourney,
    coaches=None,
    massey_snapshot=None,
)

print('m_data shape:', m_data.shape)
print('w_data shape:', w_data.shape)
display(m_data.head(3))
display(w_data.head(3))

m_data shape: (125480, 85)
w_data shape: (87734, 73)


,GameID,Gender,Season,DayNum,GameDate,GameType,isRegularSeason,isTourney,isConferenceTourney,WTeamID,WTeamName,LTeamID,LTeamName,WConfAbbrev,WConfName,LConfAbbrev,LConfName,SameConference,WSeed,WSeedNum,WSeedRegion,LSeed,LSeedNum,LSeedRegion,WLoc,NeutralSite,WHome,LHome,WAway,LAway,CityID,GameCity,GameState,CRType,WCoachName,LCoachName,WScore,LScore,ScoreDiff,TotalScore,NumOT,WFGM,WFGA,WFGM3,WFGA3,WFTM,WFTA,WOR,WDR,WAst,WTO,WStl,WBlk,WPF,LFGM,LFGA,LFGM3,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF,WHomeAway,DayZero,RegionW,RegionX,RegionY,RegionZ,WFirstD1Season,WLastD1Season,WYearsInD1,LFirstD1Season,LLastD1Season,LYearsInD1,WSeedPlayIn,LSeedPlayIn,WCoachStartDayNum,WCoachEndDayNum,LCoachStartDayNum,LCoachEndDayNum
0,M_2003_10_1104_1328,M,2003,10,2002-11-14,RegularSeason,1,0,0,1104,Alabama,1328,Oklahoma,sec,Southeastern Conference,big_twelve,Big 12 Conference,0,Y10,10,Y,W01,1,W,N,1,0,0,0,0,NaN,NaN,NaN,NaN,mark_gottfried,kelvin_sampson,68,62,6,130,0,27,58,3,14,11,18,14,24,13,23,7,1,22,22,53,2,10,16,22,10,22,8,18,9,2,20,N,2002-11-04,East,South,Midwest,West,1985,2026,18,1985,2026,18,<NA>,<NA>,0.0,154.0,0.0,154.0
1,M_2003_10_1272_1393,M,2003,10,2002-11-14,RegularSeason,1,0,0,1272,Memphis,1393,Syracuse,cusa,Conference USA,big_east,Big East Conference,0,Z07,7,Z,W03,3,W,N,1,0,0,0,0,NaN,NaN,NaN,NaN,john_calipari,jim_boeheim,70,63,7,133,0,26,62,8,20,10,19,15,28,16,13,4,4,18,24,67,6,24,9,20,20,25,7,12,8,6,16,N,2002-11-04,East,South,Midwest,West,1985,2026,18,1985,2026,18,<NA>,<NA>,0.0,154.0,0.0,154.0
2,M_2003_11_1186_1458,M,2003,11,2002-11-15,RegularSeason,1,0,0,1458,Wisconsin,1186,E Washington,big_ten,Big Ten Conference,big_sky,Big Sky Conference,0,Y05,5,Y,NaN,<NA>,<NA>,H,0,1,0,0,1,NaN,NaN,NaN,NaN,bo_ryan,ray_giacoletti,81,55,26,136,0,26,57,6,12,23,27,12,24,12,9,9,3,18,20,46,3,11,12,17,6,22,8,19,4,3,25,H,2002-11-04,East,South,Midwest,West,1985,2026,18,1985,2026,18,<NA>,<NA>,0.0,154.0,0.0,154.0


,GameID,Gender,Season,DayNum,GameDate,GameType,isRegularSeason,isTourney,isConferenceTourney,WTeamID,WTeamName,LTeamID,LTeamName,WConfAbbrev,WConfName,LConfAbbrev,LConfName,SameConference,WSeed,WSeedNum,WSeedRegion,LSeed,LSeedNum,LSeedRegion,WLoc,NeutralSite,WHome,LHome,WAway,LAway,CityID,GameCity,GameState,CRType,WScore,LScore,ScoreDiff,TotalScore,NumOT,WFGM,WFGA,WFGM3,WFGA3,WFTM,WFTA,WOR,WDR,WAst,WTO,WStl,WBlk,WPF,LFGM,LFGA,LFGM3,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF,WHomeAway,DayZero,RegionW,RegionX,RegionY,RegionZ,WSeedPlayIn,LSeedPlayIn
0,W_2010_11_3102_3394,W,2010,11,2009-11-13,RegularSeason,1,0,0,3394,TAM C. Christi,3102,Air Force,southland,Southland Conference,mwc,Mountain West Conference,0,NaN,<NA>,<NA>,NaN,<NA>,<NA>,H,0,1,0,0,1,4085.0,Corpus Christi,TX,Regular,65,46,19,111,0,25,64,2,18,13,18,21,24,13,18,10,1,16,15,47,5,17,11,20,12,21,8,25,10,0,19,H,2009-11-02,Dayton,Memphis,KansasCity,Sacramento,<NA>,<NA>
1,W_2010_11_3103_3237,W,2010,11,2009-11-13,RegularSeason,1,0,0,3103,Akron,3237,IUPUI,mac,Mid-American Conference,summit,Summit League,0,NaN,<NA>,<NA>,NaN,<NA>,<NA>,H,0,1,0,0,1,4002.0,Akron,OH,Regular,63,49,14,112,0,23,54,5,9,12,19,10,26,14,18,7,0,15,20,54,3,13,6,10,11,27,11,23,7,6,19,H,2009-11-02,Dayton,Memphis,KansasCity,Sacramento,<NA>,<NA>
2,W_2010_11_3104_3399,W,2010,11,2009-11-13,RegularSeason,1,0,0,3104,Alabama,3399,Tennessee Tech,sec,Southeastern Conference,ovc,Ohio Valley Conference,0,NaN,<NA>,<NA>,NaN,<NA>,<NA>,N,1,0,0,0,0,4085.0,Corpus Christi,TX,Regular,73,68,5,141,0,26,62,5,12,16,28,16,31,15,20,5,2,25,25,63,4,21,14,27,14,26,7,20,4,2,27,N,2009-11-02,Dayton,Memphis,KansasCity,Sacramento,<NA>,<NA>


In [13]:
full_data = pd.concat([m_data, w_data], ignore_index=True)
full_data = full_data.sort_values(['Season', 'DayNum', 'GameID']).reset_index(drop=True)

m_data.to_csv(OUT_M, index=False)
w_data.to_csv(OUT_W, index=False)
full_data.to_csv(OUT_FULL, index=False)
full_data.to_csv(OUT_FULL_COMPAT, index=False)

print('Zapisano pliki:')
print('-', OUT_M.resolve())
print('-', OUT_W.resolve())
print('-', OUT_FULL.resolve())
print('-', OUT_FULL_COMPAT.resolve())
print('full_data shape:', full_data.shape)
print('liczba kolumn:', full_data.shape[1])

Zapisano pliki:
- C:\Users\Stefan\Desktop\March-Machine-Learning-Mania-2026\data\m_data.csv
- C:\Users\Stefan\Desktop\March-Machine-Learning-Mania-2026\data\w_data.csv
- C:\Users\Stefan\Desktop\March-Machine-Learning-Mania-2026\fulldata.csv
- C:\Users\Stefan\Desktop\March-Machine-Learning-Mania-2026\fulldataset.csv
full_data shape: (213214, 85)
liczba kolumn: 85


In [14]:
summary = pd.DataFrame({
    'dataset': ['m_data', 'w_data', 'full_data'],
    'rows': [len(m_data), len(w_data), len(full_data)],
    'cols': [m_data.shape[1], w_data.shape[1], full_data.shape[1]],
    'first_season': [m_data['Season'].min(), w_data['Season'].min(), full_data['Season'].min()],
    'last_season': [m_data['Season'].max(), w_data['Season'].max(), full_data['Season'].max()],
})

display(summary)
print('\nNajważniejsze kolumny:')
print(full_data.columns.tolist())

,dataset,rows,cols,first_season,last_season
0,m_data,125480,85,2003,2026
1,w_data,87734,73,2010,2026
2,full_data,213214,85,2003,2026



Najważniejsze kolumny:
['GameID', 'Gender', 'Season', 'DayNum', 'GameDate', 'GameType', 'isRegularSeason', 'isTourney', 'isConferenceTourney', 'WTeamID', 'WTeamName', 'LTeamID', 'LTeamName', 'WConfAbbrev', 'WConfName', 'LConfAbbrev', 'LConfName', 'SameConference', 'WSeed', 'WSeedNum', 'WSeedRegion', 'LSeed', 'LSeedNum', 'LSeedRegion', 'WLoc', 'NeutralSite', 'WHome', 'LHome', 'WAway', 'LAway', 'CityID', 'GameCity', 'GameState', 'CRType', 'WCoachName', 'LCoachName', 'WScore', 'LScore', 'ScoreDiff', 'TotalScore', 'NumOT', 'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF', 'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF', 'WHomeAway', 'DayZero', 'RegionW', 'RegionX', 'RegionY', 'RegionZ', 'WFirstD1Season', 'WLastD1Season', 'WYearsInD1', 'LFirstD1Season', 'LLastD1Season', 'LYearsInD1', 'WSeedPlayIn', 'LSeedPlayIn', 'WCoachStartDayNum', 'WCoachEndDayNum', 'LCoachStartDayNum', 'LCoachEndDay